# 직접 수집 2 · 정제 — 네트워크 필터링

확장된 네트워크에는 분석에 방해되는 **노이즈 문서**가 섞여 있습니다.
- `List of …`, `Lists of …` 같은 **목록성 문서**
- `data/wiki_rule.xlsx` 에 정의된 **제외 규칙**(title/category 키워드)

무거운 XTools 수집(다음 노트북) **전에** 걸러서, 불필요한 크롤링을 줄입니다.

In [2]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # wiki_crawling.py 있는 프로젝트 루트 탐색
    if os.path.isfile("wiki_crawling.py"):
        break
    os.chdir("..")
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 2                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

작업 루트: /Users/sumin/Desktop/유망 아이템/wiki_web
대상 문서: Quantum computing | 작업 폴더: runs/Quantum_computing


In [3]:
# [이전 단계 재현] 시드 실제 제목 (캐시되어 즉시)
df_true = P.resolve_true_title(SEED, os.path.join(BASE, "true_title.xlsx"))
SEED_TRUE = P.title_space(str(df_true.loc[0, "True_title"]))
print("시드 실제 제목:", SEED_TRUE)

시드 실제 제목: Quantum computing


In [4]:
# [이전 단계 재현] 네트워크 확장 (캐시되어 즉시)
EXPAND = P.expand_network(SEED_TRUE, EXPAND, N_DEPTH)
print("확장 결과:", EXPAND)

[Quantum computing] 기존 확장 파일 발견 (alt): runs/Quantum_computing/seed_item/Quantum computing/2차시 확장 최종_결과.xlsx
확장 결과: runs/Quantum_computing/seed_item/Quantum computing/2차시 확장 최종_결과.xlsx


## 필터링 실행 — 전/후 비교

In [5]:
before = pd.read_excel(EXPAND)
before_nodes = set(before["From_title"].dropna()) | set(before["To_seealso"].dropna())

FILTER = P.filter_network(EXPAND, SEED_TRUE, FILTER, mode="balanced")

after = pd.read_excel(FILTER)
after_nodes = set(after["From_title"].dropna()) | set(after["To_seealso"].dropna())

print(f"엣지  : {len(before):3d}  →  {len(after):3d}")
print(f"노드  : {len(before_nodes):3d}  →  {len(after_nodes):3d}")
removed = before_nodes - after_nodes
print(f"제외된 문서 {len(removed)}개:")
for x in sorted(removed):
    print("  -", x)

'List of…' 문서 91개 엣지 제외
필터링 요약: 원본: 184 → 필터 후: 83 
엣지  : 184  →   83
노드  : 170  →   76
제외된 문서 94개:
  - ACM/IEEE Supercomputing Conference
  - ARPA-E
  - Advanced Research Projects Agency for Health
  - American Graphics Institute
  - Artificial Intelligence (book)
  - Bayesian approaches to brain function
  - Bell's theorem
  - Bill Gates
  - Bioethics
  - Casuistry
  - Chris Lattner
  - Computer ethics
  - Concepts, Techniques, and Models of Computer Programming
  - Core Python Programming
  - DARPA
  - David Heinemeier Hansson
  - Differential technological development
  - Diffusion of innovations
  - Disruptive innovation
  - Ecological modernization
  - Electronic Workshops in Computing
  - Engineering ethics
  - Environmental technology
  - Essentials of Programming Languages
  - Ethics of nanotechnologies
  - Frugal innovation
  - Green development
  - HSARPA
  - Hacker Manifesto
  - Handbook of Automated Reasoning
  - Head First (book series)
  - How to Design Programs
  - How 

## 남은 연관 문서 (수집 대상)

In [6]:
after.head(20)

,From_title,To_seealso,weight
0,D-Wave Systems,AQUA@home,1
1,D-Wave Systems,Adiabatic quantum computation,1
2,D-Wave Systems,Analog computer,1
3,D-Wave Systems,Flux qubit,1
4,D-Wave Systems,IBM Q System One,1
5,D-Wave Systems,Quantum annealing,1
6,D-Wave Systems,Superconducting quantum computing,1
7,Magic state distillation,Magic (quantum information),1
8,Metacomputing,Complex system,1
9,Metacomputing,Computer,1


**정리**
- 목록성/규칙 문서를 제외해 **분석 가치가 있는 노드만** 남겼습니다.
- 다음 노트북(**03 수집②**)에서 이 노드들의 편집·조회·링크·정보를 수집합니다.